In [0]:
from pyspark.sql.functions import col, when, round, current_timestamp, lit

# Read from Bronze
df_bronze = spark.table("healthcare_bronze.wait_times_raw")
print(f"Bronze records: {df_bronze.count()}")

# Transform — Silver layer
df_silver = (df_bronze
    .dropna(subset=["city", "hospital_id", "wait_time_hours"])
    .dropDuplicates(["hospital_id", "timestamp"])
    .filter(col("wait_time_hours").between(0, 72))
    .filter(col("patients_waiting") > 0)
    .withColumn("wait_category",
        when(col("wait_time_hours") < 4, "LOW")
        .when(col("wait_time_hours") < 12, "MEDIUM")
        .otherwise("HIGH")
    )
    .withColumn("wait_time_hours", round(col("wait_time_hours"), 2))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("layer", lit("silver"))
)

print(f"Silver records: {df_silver.count()}")
df_silver.show(5)

Bronze records: 100
Silver records: 100
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+-------------+--------------------+------+
|     city|hospital_id|wait_time_hours|patients_waiting|           timestamp|              source| ingestion_timestamp|       source_system|processing_date|wait_category|    silver_timestamp| layer|
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+-------------+--------------------+------+
| Winnipeg|       H003|          23.82|              26|1.7894588222486215E9|healthcare_simula...|2026-09-15 07:56:...|event_hubs_simula...|     2026-09-15|         HIGH|2026-09-16 03:11:...|silver|
| Winnipeg|       H021|           9.54|              94| 1.789458822249136E9|healthcare_simula...|2026-09-15 07:56:...|event_hubs_simula...|     2026-09-15|       M

In [0]:
# Create Silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_silver")

# Write Silver layer
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_silver.wait_times_clean")
)

print(f"✅ Silver layer written!")
print(f"Records: {spark.table('healthcare_silver.wait_times_clean').count()}")

✅ Silver layer written!
Records: 100


In [0]:
from pyspark.sql.functions import avg, count, max, min, round

# Create Gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_gold")

# Read Silver
df_silver = spark.table("healthcare_silver.wait_times_clean")

# Gold aggregations
df_gold = (df_silver
    .groupBy("city", "processing_date")
    .agg(
        round(avg("wait_time_hours"), 2).alias("avg_wait_hours"),
        round(max("wait_time_hours"), 2).alias("max_wait_hours"),
        round(min("wait_time_hours"), 2).alias("min_wait_hours"),
        count("hospital_id").alias("total_hospitals"),
        round(avg("patients_waiting"), 0).alias("avg_patients")
    )
    .withColumn("gold_timestamp", current_timestamp())
)

# Write Gold layer
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_gold.city_metrics")
)

print("✅ Gold layer written!")
df_gold.show()

✅ Gold layer written!
+---------+---------------+--------------+--------------+--------------+---------------+------------+--------------------+
|     city|processing_date|avg_wait_hours|max_wait_hours|min_wait_hours|total_hospitals|avg_patients|      gold_timestamp|
+---------+---------------+--------------+--------------+--------------+---------------+------------+--------------------+
| Winnipeg|     2026-09-15|         12.11|         23.82|          1.04|             25|       112.0|2026-09-16 03:15:...|
|  Calgary|     2026-09-15|         12.52|         21.88|          4.89|             13|       105.0|2026-09-16 03:15:...|
|  Toronto|     2026-09-15|         13.45|         23.85|          1.89|             22|       116.0|2026-09-16 03:15:...|
|  Halifax|     2026-09-15|          9.19|         21.37|          1.17|             24|       111.0|2026-09-16 03:15:...|
|Vancouver|     2026-09-15|         14.24|         23.16|          3.07|             16|       108.0|2026-09-16 03:15

In [0]:
spark.table("healthcare_gold.city_metrics").show()

+---------+---------------+--------------+--------------+--------------+---------------+------------+--------------------+
|     city|processing_date|avg_wait_hours|max_wait_hours|min_wait_hours|total_hospitals|avg_patients|      gold_timestamp|
+---------+---------------+--------------+--------------+--------------+---------------+------------+--------------------+
| Winnipeg|     2026-09-15|         12.11|         23.82|          1.04|             25|       112.0|2026-09-16 03:14:...|
|  Calgary|     2026-09-15|         12.52|         21.88|          4.89|             13|       105.0|2026-09-16 03:14:...|
|  Toronto|     2026-09-15|         13.45|         23.85|          1.89|             22|       116.0|2026-09-16 03:14:...|
|  Halifax|     2026-09-15|          9.19|         21.37|          1.17|             24|       111.0|2026-09-16 03:14:...|
|Vancouver|     2026-09-15|         14.24|         23.16|          3.07|             16|       108.0|2026-09-16 03:14:...|
+---------+-----